# NPOV Drift Detection — Notebook 2: the metadata-only null model

**Task:** [T428896 — WikiNPOV Drift Detector](https://phabricator.wikimedia.org/T428896)
**Input:** the event and control sets built by [Notebook 1](01_ground_truth.ipynb).
Read-only, public API, no article text, no ML models downloaded.

## The question

Before testing stance balance, perspective clusters or embedding drift, we need to know what
**edit metadata alone** already predicts. Nobody has established this, and without it every content
signal reports an uninterpretable number: "AUC 0.88" means nothing if revision count alone gets 0.85.

So: using only who edited, how often, how much they changed and how much got reverted — no article
text whatsoever — how well can we separate *an article about to be tagged NPOV* from *a matched
article that never was*?

This notebook deliberately contains no neutrality modelling at all. It is the floor.

## Why this is the decision point

The answer reshapes the project:

- **If metadata alone separates them well**, then "detecting NPOV drift" is substantially "detecting
  contested editing", which is already what [ORES/Lift Wing](https://www.mediawiki.org/wiki/ORES)
  and edit-war research do. The contribution then has to be **lead time** and **cross-lingual
  transfer**, and the content signals are an ablation against this floor.
- **If metadata alone does poorly**, the content signals have real room — and the prototype's
  100% control false-positive rate was a controls problem, not a signal problem.

Either way we learn something, and we stop reporting numbers without a floor under them.

## Design decisions, stated before looking at any result

**Window.** Features are computed over the 90 days **strictly before** the onset revision. The onset
revision itself is excluded — it is an edit by an editor and would leak the label.

**Control windows are calendar-matched.** Each control uses the window ending at *its matched
positive's* onset timestamp. Wiki-wide editing rates drift over the years, so an unmatched window
would let the model separate classes on "what year is it".

**No current-state features.** Tempting metadata like a user's `editcount` reflects edits made
*after* our window, so it is excluded. Account **registration date** is static and safe, so
new-account share is allowed.

**Pair-grouped cross-validation.** A positive and its matched control are one group and never
straddle a train/test split. Otherwise the model can memorise the topic.

**Pre-registered reading of the result**, so the conclusion is not chosen after the fact:

| Mean CV ROC-AUC | Reading |
|---|---|
| ≥ 0.80 | metadata dominates; re-scope the project to lead time + cross-lingual transfer |
| 0.65 – 0.80 | partial; content signals must be reported as a *delta over this floor*, never alone |
| ≤ 0.65 | metadata is weak; content signals have real room, and NB1's matched controls were the missing piece |

In [ ]:
# --- Configuration -------------------------------------------------------------
WIKI_USERNAME = "MahimaSinghal"
LANG          = "en"

EVENTS_CSV   = "npov_data/full_npov_events_en.csv"      # from Notebook 1
CONTROLS_CSV = "npov_data/full_npov_controls_en.csv"
WINDOW_DAYS  = 90
NEW_ACCT_DAYS = 30        # "new account" = registered within this many days of the window end
RANDOM_SEED  = 20261017
CACHE_DIR    = "npov_cache"
OUT_DIR      = "npov_data"
USE_PAGEVIEWS = True      # one extra REST call per article; set False to skip

import os, re, json, time, math, random, hashlib, datetime, urllib.parse
from collections import Counter
import requests
import numpy as np
import pandas as pd

os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

API = "https://" + LANG + ".wikipedia.org/w/api.php"
UA  = ("npov-drift-research/0.2 (https://phabricator.wikimedia.org/T428896; "
       "User:" + WIKI_USERNAME + ") python-requests")
print("API:", API)

In [ ]:
# --- Cached, rate-limited clients ----------------------------------------------
SESSION = requests.Session(); SESSION.headers.update({"User-Agent": UA})
_last = [0.0]; MIN_INTERVAL = 0.12
stats = {"calls": 0, "hits": 0, "retries": 0}

def _cached_get(url, params, tag):
    key = hashlib.sha1((tag + url + json.dumps(params, sort_keys=True, default=str)).encode()).hexdigest()
    path = os.path.join(CACHE_DIR, key + ".json")
    if os.path.exists(path):
        stats["hits"] += 1
        with open(path) as f:
            return json.load(f)
    # Transient failures MUST be retried, never cached and never allowed to become a
    # feature value. In an earlier run, ~54% of control pageview calls failed transiently,
    # the exception handler turned each into NaN, and "pageviews is missing" became a
    # spurious 32x class separator that inflated the headline AUC from 0.76 to 0.91.
    # A fake signal manufactured out of API flakiness is the exact failure this whole
    # notebook exists to prevent, so the retry below is load-bearing, not politeness.
    data = None
    for attempt in range(4):
        w = MIN_INTERVAL - (time.time() - _last[0])
        if w > 0:
            time.sleep(w)
        try:
            r = SESSION.get(url, params=params, timeout=60)
        except requests.RequestException:
            _last[0] = time.time()
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
            continue
        _last[0] = time.time(); stats["calls"] += 1
        if r.status_code == 404:
            data = {"__http404": True}          # a real "no such article/range"
            break
        if r.status_code in (429, 500, 502, 503, 504):
            stats["retries"] += 1
            if attempt == 3:
                r.raise_for_status()
            time.sleep(2 * 2 ** attempt)        # back off and try again
            continue
        r.raise_for_status()
        data = r.json()
        break
    with open(path, "w") as f:
        json.dump(data, f)
    return data

def api(**params):
    params.setdefault("action", "query")
    params.setdefault("format", "json")
    params.setdefault("formatversion", "2")
    d = _cached_get(API, params, "api")
    if "error" in d:
        raise RuntimeError(d["error"])
    return d

def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

def to_utc(s):
    """Parse timestamps that may mix 'Z' and '+00:00' forms, across pandas versions."""
    try:
        return pd.to_datetime(s, utc=True, format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(s, utc=True)

print("client ready")

In [ ]:
# --- Load Notebook 1's output --------------------------------------------------
ev = pd.read_csv(EVENTS_CSV)
ev = ev[ev.status == "located"].copy()
ev["onset_ts"] = to_utc(ev.onset_ts)
ct = pd.read_csv(CONTROLS_CSV)

# each control inherits the calendar window of the positive it was matched to
ct = ct.merge(ev[["title", "onset_ts"]].rename(columns={"title": "matched_to",
                                                        "onset_ts": "window_end_ts"}),
              on="matched_to", how="inner")

pairs = pd.concat([
    ev.assign(y=1, pair=ev.title, window_end_ts=ev.onset_ts)[
        ["title", "y", "pair", "window_end_ts"]],
    ct.assign(y=0, pair=ct.matched_to)[["title", "y", "pair", "window_end_ts"]],
], ignore_index=True)

# keep only complete pairs, so the design stays balanced
good = pairs.groupby("pair").y.agg(["sum", "count"])
good = good[(good["sum"] == 1) & (good["count"] == 2)].index
pairs = pairs[pairs.pair.isin(good)].reset_index(drop=True)

print("%d located positives, %d controls loaded, %d complete pairs kept"
      % (len(ev), len(ct), len(good)))
print(pairs.y.value_counts().to_string())

## Features

Sixteen metadata features over the 90-day window, plus talk-page activity, new-account share and
pageviews. Nothing here reads a single word of the article body.

In [ ]:
REVERTING_TAGS = {"mw-undo", "mw-rollback", "mw-manual-revert"}

def revisions_in(title, start, end):
    """Revisions with start <= ts < end, oldest first."""
    out, cont = [], {}
    while True:
        d = api(prop="revisions", titles=title,
                rvprop="ids|timestamp|user|comment|size|tags|flags",
                rvlimit=500, rvdir="newer", rvstart=iso(start), rvend=iso(end), **cont)
        pages = d.get("query", {}).get("pages", [])
        if not pages or pages[0].get("missing"):
            return []
        out += pages[0].get("revisions", [])
        if "continue" not in d:
            break
        cont = d["continue"]
    cut = iso(end)
    return [r for r in out if r["timestamp"] < cut]

_reg_cache = {}
def registrations(users):
    """Account registration timestamps, batched 50 at a time and memoised."""
    want = sorted({u for u in users if u and not re.match(r"^\d+\.|:", u) and u not in _reg_cache})
    for i in range(0, len(want), 50):
        chunk = want[i:i + 50]
        d = api(list="users", ususers="|".join(chunk), usprop="registration")
        for u in d.get("query", {}).get("users", []):
            _reg_cache[u.get("name")] = u.get("registration")
    return {u: _reg_cache.get(u) for u in users}

def pageviews_sum(title, start, end):
    if not USE_PAGEVIEWS:
        return np.nan
    url = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           + LANG + ".wikipedia/all-access/user/"
           + urllib.parse.quote(title.replace(" ", "_"), safe="")
           + "/daily/" + start.strftime("%Y%m%d") + "/" + end.strftime("%Y%m%d"))
    try:
        d = _cached_get(url, {}, "pv")
    except Exception:
        return np.nan
    if d.get("__http404") or "items" not in d:
        return np.nan
    return float(sum(i.get("views", 0) for i in d["items"]))

def features(title, end_ts, days=WINDOW_DAYS):
    start = end_ts - datetime.timedelta(days=days)
    revs = revisions_in(title, start, end_ts)
    talk = revisions_in("Talk:" + title, start, end_ts)
    f = {"n_revs": float(len(revs)), "talk_revs": float(len(talk)),
         "empty_window": 1.0 if not revs else 0.0,
         "pageviews": pageviews_sum(title, start, end_ts)}
    if not revs:
        return f
    users = [r.get("user") or "?" for r in revs]
    counts = Counter(users)
    n = float(len(revs))
    shares = np.array([c / n for c in counts.values()])
    sizes = [float(r.get("size") or 0) for r in revs]
    deltas = np.diff(sizes) if len(sizes) > 1 else np.array([0.0])
    tagsets = [set(r.get("tags") or []) for r in revs]
    reg = registrations(users)
    cutoff = end_ts - datetime.timedelta(days=NEW_ACCT_DAYS)

    def is_new(u):
        t = reg.get(u)
        if not t:
            return False
        try:
            return pd.to_datetime(t, utc=True) >= cutoff
        except Exception:
            return False

    f.update({
        "n_editors": float(len(counts)),
        "editor_hhi": float((shares ** 2).sum()),
        "top_editor_share": float(shares.max()),
        "anon_share": float(np.mean([1.0 if r.get("anon") else 0.0 for r in revs])),
        "bot_share": float(np.mean([1.0 if re.search(r"bot\b", u, re.I) else 0.0 for u in users])),
        "minor_share": float(np.mean([1.0 if r.get("minor") else 0.0 for r in revs])),
        "reverting_share": float(np.mean([1.0 if (t & REVERTING_TAGS) else 0.0 for t in tagsets])),
        "reverted_share": float(np.mean([1.0 if "mw-reverted" in t else 0.0 for t in tagsets])),
        "new_account_edit_share": float(np.mean([1.0 if is_new(u) else 0.0 for u in users])),
        "revs_per_day": float(n / days),
        "max_revs_one_day": float(max(Counter(r["timestamp"][:10] for r in revs).values())),
        "size_end": sizes[-1],
        "size_net_delta": sizes[-1] - sizes[0],
        "size_abs_churn": float(np.abs(deltas).sum()),
        "size_delta_std": float(np.std(deltas)),
        "empty_summary_share": float(np.mean([1.0 if not (r.get("comment") or "").strip() else 0.0
                                              for r in revs])),
    })
    return f

print("feature functions defined")

In [ ]:
# --- Build the feature matrix --------------------------------------------------
rows, t0 = [], time.time()
for i, r in pairs.iterrows():
    try:
        f = features(r.title, r.window_end_ts.to_pydatetime())
    except Exception as e:
        f = {"error": type(e).__name__}
    rows.append({"title": r.title, "y": r.y, "pair": r.pair,
                 "window_end_ts": r.window_end_ts, **f})
    if (i + 1) % 20 == 0 or i + 1 == len(pairs):
        el = time.time() - t0
        print("  %d/%d  %5.0fs  (%.1f s/article, %d live calls)"
              % (i + 1, len(pairs), el, el / (i + 1), stats["calls"]))

X = pd.DataFrame(rows)
X.to_csv(os.path.join(OUT_DIR, "nb2_features.csv"), index=False)
print("\nfeature matrix:", X.shape)
print("rows with an error:", X.get("error", pd.Series(dtype=object)).notna().sum())

### Sanity gate on the features themselves

Before any model sees these columns: a feature whose **missingness** differs by class is a fake
signal, not a measurement. That is how an earlier run of this notebook reported 0.91 instead of
0.76 — transient pageview failures were concentrated in the control rows, so "value is missing"
became a near-perfect label proxy. This cell fails loudly on that pattern instead of modelling it.

In [ ]:
RAW = [c for c in X.columns if c not in ("title", "y", "pair", "window_end_ts", "error")]
miss = (X.assign(**{c + "__m": X[c].isna() for c in RAW})
         .groupby("y")[[c + "__m" for c in RAW]].mean().T)
miss.columns = ["control", "tagged"]
miss["gap"] = (miss.tagged - miss.control).abs()
bad = miss[miss.gap > 0.10].sort_values("gap", ascending=False)
if len(bad):
    print("!! WARNING - class-dependent missingness, do NOT model these as-is:\n")
    print(bad.to_string(float_format=lambda v: "%.3f" % v))
    print("\nRe-run the affected rows; a retry usually resolves it. If the gap is real and not")
    print("an API artifact, drop the feature rather than imputing it.")
else:
    print("OK: no feature has a class-dependent missingness gap above 10pp")
    print("max gap: %.3f (%s)" % (miss.gap.max(), miss.gap.idxmax()))
print("\ntransient failures retried during this run:", stats["retries"])

In [ ]:
FEATURES = ["n_revs", "revs_per_day", "max_revs_one_day", "n_editors", "editor_hhi",
            "top_editor_share", "anon_share", "bot_share", "minor_share",
            "reverting_share", "reverted_share", "new_account_edit_share",
            "size_end", "size_net_delta", "size_abs_churn", "size_delta_std",
            "empty_summary_share", "talk_revs", "pageviews", "empty_window"]
FEATURES = [f for f in FEATURES if f in X.columns]

D = X.dropna(subset=["y"]).copy()
for c in FEATURES:
    D[c] = pd.to_numeric(D[c], errors="coerce")
# median-impute, and keep an explicit missingness flag for pageviews
if "pageviews" in FEATURES:
    D["pageviews_missing"] = D.pageviews.isna().astype(float)
    FEATURES = FEATURES + ["pageviews_missing"]
D[FEATURES] = D[FEATURES].fillna(D[FEATURES].median())

# drop pairs that lost a side
cnt = D.groupby("pair").y.agg(["sum", "count"])
keep = cnt[(cnt["sum"] == 1) & (cnt["count"] == 2)].index
D = D[D.pair.isin(keep)].reset_index(drop=True)
print("modelling set: %d rows, %d pairs, %d features" % (len(D), D.pair.nunique(), len(FEATURES)))
print("\nclass means (raw, unstandardised):")
print(D.groupby("y")[FEATURES].mean().T.to_string(float_format=lambda v: "%.3f" % v))

## The floor

Two models, both on the same pair-grouped splits: logistic regression (a linear read on the
features) and gradient boosting (to catch interactions). Reported with a bootstrap confidence
interval, because with a few hundred pairs a point estimate of AUC is not a number you can act on.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

def cv_scores(model, D, features, n_splits=5, seed=RANDOM_SEED):
    y = D.y.values.astype(int)
    g = D.pair.values
    Xm = D[features].values
    oof = np.full(len(D), np.nan)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_auc = []
    for tr, te in cv.split(Xm, y, groups=g):
        m = model()
        m.fit(Xm[tr], y[tr])
        p = m.predict_proba(Xm[te])[:, 1]
        oof[te] = p
        if len(set(y[te])) == 2:
            fold_auc.append(roc_auc_score(y[te], p))
    return oof, np.array(fold_auc)

def boot_ci(y, p, n=2000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y)); out = []
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        if len(set(y[s])) == 2:
            out.append(roc_auc_score(y[s], p[s]))
    return float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))

MODELS = {
    "logistic": lambda: make_pipeline(StandardScaler(),
                                      LogisticRegression(max_iter=2000, C=1.0)),
    # min_samples_leaf below sklearn's default of 20: with a few hundred pairs the
    # default makes every split illegal and the model predicts a constant.
    "grad boosting": lambda: HistGradientBoostingClassifier(
        max_depth=3, max_iter=250, learning_rate=0.06, min_samples_leaf=10,
        random_state=RANDOM_SEED),
}

results = {}
yv = D.y.values.astype(int)
for name, ctor in MODELS.items():
    oof, folds = cv_scores(ctor, D, FEATURES)
    auc = roc_auc_score(yv, oof)
    lo, hi = boot_ci(yv, oof)
    ap = average_precision_score(yv, oof)
    results[name] = {"auc": auc, "ci": (lo, hi), "folds": folds, "ap": ap, "oof": oof}
    print("%-14s ROC-AUC %.3f  [95%% CI %.3f-%.3f]   per-fold %s   PR-AUC %.3f"
          % (name, auc, lo, hi, np.round(folds, 3), ap))

### Sensitivity check: is the floor just "something happened"?

If a control article was dormant during its window while the positive was being actively edited,
any model separates them on activity alone, and that tells us nothing about neutrality. Notebook 1
screens controls for window activity, but older control sets may not, so the floor is recomputed
here on the pairs where **both** sides saw at least one edit. The gap between the two numbers is how
much of the floor was activity-versus-dormancy.

In [ ]:
active_pairs = (D.assign(act=(D.n_revs > 0).astype(int))
                 .groupby("pair").act.min())
keep_act = set(active_pairs[active_pairs == 1].index)
Da = D[D.pair.isin(keep_act)].reset_index(drop=True)
print("pairs where both sides were edited in the window: %d of %d"
      % (Da.pair.nunique(), D.pair.nunique()))

if Da.pair.nunique() >= 20:
    ya = Da.y.values.astype(int)
    for name, ctor in MODELS.items():
        oof_a, folds_a = cv_scores(ctor, Da, FEATURES,
                                   n_splits=min(5, Da.pair.nunique() // 4))
        lo, hi = boot_ci(ya, oof_a)
        print("  %-14s ROC-AUC %.3f  [95%% CI %.3f-%.3f]  (both-active subset)"
              % (name, roc_auc_score(ya, oof_a), lo, hi))
else:
    print("too few both-active pairs to recompute; skipping")

In [ ]:
# How much of that is one feature? Univariate AUC, direction-corrected.
uni = []
for f in FEATURES:
    v = D[f].values
    if np.nanstd(v) == 0:
        continue
    a = roc_auc_score(yv, v)
    uni.append({"feature": f, "auc": max(a, 1 - a),
                "direction": "higher in tagged" if a > 0.5 else "higher in control"})
uni = pd.DataFrame(uni).sort_values("auc", ascending=False).reset_index(drop=True)
print("Single-feature AUC (what one metadata column achieves alone):\n")
print(uni.to_string(index=False, float_format=lambda v: "%.3f" % v))

### Nested specifications: which number is actually the floor?

The headline AUC above is an **upper bound**, not the floor, because two of its features are
contaminated:

- **`pageviews` / `pageviews_missing`.** The pageviews REST endpoint 404s on obscure articles, and
  NPOV-tagged articles are far more popular than their controls (in this run, ~29k views per 90 days
  against ~3k). Controls were matched on length and revision count, never on traffic, so this is a
  confound matching never addressed. `pageviews_missing` alone reaches AUC 0.76 — it is an obscurity
  detector.
- **`empty_window`.** Notebook 1 screens *controls* for window activity but not positives, so an
  empty window is near-diagnostic of a positive purely by construction.

So the floor is reported as a ladder, each rung stripping something that might be doing the work for
the wrong reason. The honest number to hold later signals against is **C**.

In [ ]:
SOCIAL = ["editor_hhi", "top_editor_share", "anon_share", "bot_share", "minor_share",
          "reverting_share", "reverted_share", "new_account_edit_share", "empty_summary_share"]
NO_ARTIFACT = [f for f in FEATURES if f not in ("pageviews", "pageviews_missing", "empty_window")]
SOCIAL = [f for f in SOCIAL if f in FEATURES]

act = D.assign(_a=(D.n_revs > 0).astype(int)).groupby("pair")._a.min()
Dact = D[D.pair.isin(set(act[act == 1].index))]

def spec(df, feats, label):
    if df.pair.nunique() < 20 or not feats:
        print("  %-50s skipped (too few pairs)" % label)
        return None
    row = {"spec": label, "n_pairs": int(df.pair.nunique())}
    for name, ctor in MODELS.items():
        oof, _ = cv_scores(ctor, df, feats, n_splits=min(5, df.pair.nunique() // 4))
        row[name] = round(roc_auc_score(df.y.values.astype(int), oof), 3)
    print("  %-50s pairs=%3d  %s" % (label, row["n_pairs"],
          "  ".join("%s %.3f" % (k, row[k]) for k in MODELS)))
    return row

print("NESTED SPECIFICATIONS\n")
ladder = [r for r in [
    spec(D,    FEATURES,    "A. all features, all pairs (upper bound)"),
    spec(Dact, FEATURES,    "B. all features, both-active pairs"),
    spec(Dact, NO_ARTIFACT, "C. no traffic, no empty_window  <-- THE FLOOR"),
    spec(Dact, SOCIAL,      "D. social features only (no activity volume)"),
    spec(Dact, ["n_revs"],  "E. revision count alone (dumbest baseline)"),
    spec(Dact, ["talk_revs"], "F. talk-page revisions alone"),
] if r]
pd.DataFrame(ladder).to_csv(os.path.join(OUT_DIR, "nb2_specification_ladder.csv"), index=False)

floor = max(v for k, v in (ladder[2] if len(ladder) > 2 else {}).items()
            if k in MODELS) if len(ladder) > 2 else float("nan")
print("\nFLOOR (spec C, best model): %.3f" % floor)
print("Any content signal must be reported as a delta over THIS number, on these pairs.")

In [ ]:
# Precision@k: the deployable framing. A reviewer opens the top k% of the queue.
best = max(results, key=lambda k: results[k]["auc"])
p = results[best]["oof"]
print("ranking by the %s model's out-of-fold score\n" % best)
print("  k        precision   (base rate %.2f)" % yv.mean())
for frac in (0.05, 0.10, 0.20, 0.50):
    k = max(1, int(round(frac * len(p))))
    top = np.argsort(-p)[:k]
    print("  top %3.0f%%   %.3f" % (100 * frac, yv[top].mean()))

In [ ]:
# Temporal holdout: train on older disputes, test on the most recent fifth.
Dt = D.copy()
Dt["t"] = to_utc(Dt.window_end_ts)
pair_t = Dt.groupby("pair").t.min().sort_values()
cut = pair_t.quantile(0.80)
tr_pairs = set(pair_t[pair_t <= cut].index); te_pairs = set(pair_t[pair_t > cut].index)
tr = Dt.pair.isin(tr_pairs).values; te = Dt.pair.isin(te_pairs).values

print("train pairs %d (to %s), test pairs %d (after)"
      % (len(tr_pairs), cut.date(), len(te_pairs)))
for name, ctor in MODELS.items():
    m = ctor(); m.fit(Dt.loc[tr, FEATURES].values, Dt.loc[tr, "y"].values.astype(int))
    pt = m.predict_proba(Dt.loc[te, FEATURES].values)[:, 1]
    yt = Dt.loc[te, "y"].values.astype(int)
    if len(set(yt)) == 2:
        print("  %-14s future-year ROC-AUC %.3f" % (name, roc_auc_score(yt, pt)))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for name in results:
    fpr, tpr, _ = roc_curve(yv, results[name]["oof"])
    ax.plot(fpr, tpr, label="%s (AUC %.3f)" % (name, results[name]["auc"]))
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="chance")
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("Metadata-only floor"); ax.legend(frameon=False, fontsize=8)

ax = axes[1]
top = uni.head(10).iloc[::-1]
ax.barh(top.feature, top.auc, color="#3366cc")
ax.axvline(0.5, color="k", lw=0.8, ls="--")
ax.set_xlim(0.45, max(0.75, top.auc.max() + 0.03))
ax.set_xlabel("single-feature AUC"); ax.set_title("What one column achieves alone")
for a in axes:
    a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# --- Save ---------------------------------------------------------------------
summary = {
    "task": "https://phabricator.wikimedia.org/T428896",
    "notebook": "02_null_model",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "wiki": LANG + ".wikipedia.org",
    "window_days": WINDOW_DAYS,
    "n_pairs": int(D.pair.nunique()),
    "n_rows": int(len(D)),
    "n_features": len(FEATURES),
    "features": FEATURES,
    "models": {k: {"roc_auc": round(v["auc"], 4),
                   "ci95": [round(v["ci"][0], 4), round(v["ci"][1], 4)],
                   "pr_auc": round(v["ap"], 4),
                   "fold_aucs": [round(float(x), 4) for x in v["folds"]]}
               for k, v in results.items()},
    "top_univariate": uni.head(8).to_dict("records"),
    "live_api_calls": stats["calls"],
}
with open(os.path.join(OUT_DIR, "nb2_results.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary["models"], indent=2))

best_auc = max(v["auc"] for v in results.values())
verdict = ("metadata DOMINATES - re-scope to lead time + cross-lingual transfer" if best_auc >= 0.80
           else "PARTIAL - content signals must be reported as a delta over this floor"
           if best_auc > 0.65 else
           "metadata is WEAK - content signals have real room")
print("\nbest metadata-only AUC: %.3f" % best_auc)
print("pre-registered reading:", verdict)

## Reading the result

Whatever the number above is, it is now **the floor**. Any content signal — stance balance,
perspective clusters, due-weight shares — has to be reported as a delta over it, on these same pairs
and these same splits. "AUC 0.88 on a 20-vs-20 split" is no longer a sentence this project can emit.

**Limitations that matter for how far to trust it:**

1. **The label is human attention.** A positive is an article someone tagged, not an article that
   provably drifted. A model that predicts tagging may be predicting the arrival of an attentive
   editor. NB1's limitations section covers this; separating "gradual drift" from "edit war" and
   "one bad edit" needs the talk-page threads, which is the next piece of label work.
2. **Controls are screened, not certified.** NB1 screens candidates for any neutrality tag in their
   revision summaries — and NB1 itself measured that scan as catching only about half of real tag
   events. So some controls are probably quietly contaminated, which **depresses** the measured AUC.
   The floor reported here is therefore a lower bound.
3. **Matched on length and revision count by construction.** Features correlated with those two were
   partly equalised on purpose. That is the point — it is what the prototype's unmatched controls
   failed to do — but it means this AUC is not comparable to numbers from unmatched designs.
4. **Reverts come from change tags**, which were only rolled out progressively; `mw-reverted` in
   particular is not reliable for older revisions. Treat the revert features as weaker before ~2019.

## Next

NB3 takes the one signal that showed promise in the prototype — stance balance, via NLI — and runs it
on exactly these pairs and splits, reporting the delta over this floor. Then NB4 scores PELT on
onset *localisation* against a random-revision null, and NB5 asks whether any of it survives on
Indic-language wikis.